# Day 011 | OOP — Inheritance & Polymorphism

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals

**Author:** Ripon Chandra Malo  
**Date:** February 27, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Inheritance | A child class inherits attributes/methods from a parent | `SPTTest` inherits from `GeotechTest` |
| `super().__init__()` | Call the parent’s constructor from the child | Set common fields (depth, borehole) in the base class |
| Method overriding | Child replaces a parent method with its own version | `SPTTest.analyze()` vs `CPTTest.analyze()` |
| Polymorphism | Same method name, different behavior per class | Call `.classify()` on any test — each responds differently |
| `isinstance()` | Check if an object belongs to a class (or its parent) | Filter only triaxial tests from a mixed list |
| Multiple inheritance | A class inherits from two or more parents | `SiteReport` is both `Printable` and `Analyzable` |
| Unified interface | Process different objects through one loop | Generate a site report from SPT, CPT, and triaxial data |

---

## Our Geotechnical Problem

A site investigation produces **different types of tests**:

- **SPT** (Standard Penetration Test) — records blow count N at each depth
- **CPT** (Cone Penetration Test) — records tip resistance q_c and sleeve friction f_s
- **Triaxial** — records confining pressure σ₃, deviator stress, cohesion c, friction angle φ

Each test has **common fields** (depth, borehole ID, date) and **unique analysis methods**. Without inheritance, we would duplicate the common logic in every class. With inheritance, we write it once in a `GeotechTest` base class and let each child class add only what is unique.

**Goal:** Build a site investigation system where all test types share a common interface, can be processed in a single loop, and produce a unified site report.

---

## Part 1: Inheritance — The Base Class `GeotechTest`

**Inheritance** lets a child class reuse attributes and methods from a parent class, then add or modify behavior.

```
GeotechTest          ← Parent (base) class
├── SPTTest           ← Child: inherits + adds N-value logic
├── CPTTest           ← Child: inherits + adds cone data logic
└── TriaxialTest      ← Child: inherits + adds shear strength logic
```

Syntax: `class ChildClass(ParentClass):`

The child **inherits** everything from the parent and can:
1. Use parent methods as-is
2. **Override** parent methods with its own version
3. **Add** entirely new methods

In [ ]:
# ─────────────────────────────────────────────────────
# BASE CLASS — GeotechTest (the parent)
# ─────────────────────────────────────────────────────

class GeotechTest:
    """Base class for all geotechnical field/lab tests."""

    def __init__(self, borehole_id, depth, date="2026-02-27"):
        self.borehole_id = borehole_id
        self.depth = depth          # metres below ground
        self.date = date

    def summary(self):
        """One-line summary — inherited by all children."""
        return f"{self.borehole_id} @ {self.depth}m ({self.date})"

    def analyze(self):
        """Placeholder — each child will override this."""
        return "No analysis defined for base GeotechTest."

    def classify(self):
        """Placeholder — each child will override this."""
        return "Unclassified"


# Try the base class
test = GeotechTest("BH-01", 6.0)
print(f"Summary:  {test.summary()}")
print(f"Analyze:  {test.analyze()}")
print(f"Classify: {test.classify()}")
print(f"Type:     {type(test)}")

---

## Part 2: Child Classes & `super().__init__()` — SPTTest

A child class calls `super().__init__()` to run the parent’s constructor first, then adds its own attributes.

```python
class SPTTest(GeotechTest):            # inherits from GeotechTest
    def __init__(self, borehole_id, depth, n_value, ...):
        super().__init__(borehole_id, depth)  # parent handles common fields
        self.n_value = n_value                # child adds SPT-specific field
```

Without `super().__init__()`, the child would **not** have `borehole_id`, `depth`, or `date`.

In [ ]:
# ─────────────────────────────────────────────────────
# CHILD CLASS 1 — SPTTest (Standard Penetration Test)
# ─────────────────────────────────────────────────────

class SPTTest(GeotechTest):
    """Standard Penetration Test — records blow count N."""

    def __init__(self, borehole_id, depth, n_value, energy_ratio=0.6, date="2026-02-27"):
        super().__init__(borehole_id, depth, date)   # parent sets common fields
        self.n_value = n_value                        # raw field blow count
        self.energy_ratio = energy_ratio              # hammer energy correction

    def corrected_n(self):
        """N60 = N * (energy_ratio / 0.60)"""
        return round(self.n_value * (self.energy_ratio / 0.60))

    def estimate_vs(self):
        """Shear wave velocity from Imai & Tonouchi (1982): Vs = 97 * N60^0.314"""
        n60 = max(self.corrected_n(), 1)
        return round(97 * (n60 ** 0.314), 1)

    def analyze(self):
        """Override parent — SPT-specific analysis."""
        n60 = self.corrected_n()
        vs = self.estimate_vs()
        return {"N60": n60, "Vs": vs, "depth": self.depth}

    def classify(self):
        """Override parent — classify relative density from N60."""
        n60 = self.corrected_n()
        if n60 < 4:  return "Very Loose"
        elif n60 < 10: return "Loose"
        elif n60 < 30: return "Medium Dense"
        elif n60 < 50: return "Dense"
        else: return "Very Dense"


# Create an SPT test
spt = SPTTest("BH-01", 6.0, n_value=18, energy_ratio=0.6)

# Inherited from parent
print(f"Summary (inherited): {spt.summary()}")
print(f"Borehole ID:         {spt.borehole_id}")
print(f"Depth:               {spt.depth}m")
print()

# SPT-specific
print(f"Raw N:       {spt.n_value}")
print(f"N60:         {spt.corrected_n()}")
print(f"Vs:          {spt.estimate_vs()} m/s")
print(f"Classify:    {spt.classify()}")
print(f"Analyze:     {spt.analyze()}")

---

## Part 3: More Child Classes — CPTTest & TriaxialTest

Each child class inherits the **same base** but overrides `analyze()` and `classify()` with **completely different logic**. This is the power of inheritance — shared structure, specialized behavior.

In [ ]:
# ─────────────────────────────────────────────────────
# CHILD CLASS 2 — CPTTest (Cone Penetration Test)
# ─────────────────────────────────────────────────────

class CPTTest(GeotechTest):
    """Cone Penetration Test — records tip resistance and sleeve friction."""

    def __init__(self, borehole_id, depth, qc, fs, date="2026-02-27"):
        super().__init__(borehole_id, depth, date)
        self.qc = qc    # tip resistance (MPa)
        self.fs = fs    # sleeve friction (kPa)

    def friction_ratio(self):
        """Rf = (fs / qc) * 100, where qc converted to kPa."""
        qc_kpa = self.qc * 1000  # MPa to kPa
        return round((self.fs / qc_kpa) * 100, 2) if qc_kpa > 0 else 0

    def analyze(self):
        """Override parent — CPT-specific analysis."""
        rf = self.friction_ratio()
        return {"qc_MPa": self.qc, "fs_kPa": self.fs, "Rf_%": rf, "depth": self.depth}

    def classify(self):
        """Override parent — simplified Robertson classification."""
        rf = self.friction_ratio()
        if self.qc > 10 and rf < 1:
            return "Sand (dense)"
        elif self.qc > 5 and rf < 2:
            return "Sand"
        elif rf < 4:
            return "Silty Sand / Sandy Silt"
        elif rf < 6:
            return "Clay"
        else:
            return "Organic / Sensitive Clay"


# ─────────────────────────────────────────────────────
# CHILD CLASS 3 — TriaxialTest (Consolidated Undrained)
# ─────────────────────────────────────────────────────

class TriaxialTest(GeotechTest):
    """Consolidated Undrained Triaxial Test — records shear strength params."""

    def __init__(self, borehole_id, depth, sigma3, deviator_stress,
                 cohesion, friction_angle, date="2026-02-27"):
        super().__init__(borehole_id, depth, date)
        self.sigma3 = sigma3                    # confining pressure (kPa)
        self.deviator_stress = deviator_stress  # (sigma1 - sigma3) at failure (kPa)
        self.cohesion = cohesion                # c (kPa)
        self.friction_angle = friction_angle    # phi (degrees)

    def sigma1(self):
        """Major principal stress at failure."""
        return self.sigma3 + self.deviator_stress

    def shear_strength(self, normal_stress):
        """Mohr-Coulomb: tau = c + sigma_n * tan(phi)."""
        import math
        phi_rad = math.radians(self.friction_angle)
        return round(self.cohesion + normal_stress * math.tan(phi_rad), 1)

    def analyze(self):
        """Override parent — triaxial-specific analysis."""
        return {
            "sigma3_kPa": self.sigma3,
            "sigma1_kPa": self.sigma1(),
            "c_kPa": self.cohesion,
            "phi_deg": self.friction_angle,
            "depth": self.depth,
        }

    def classify(self):
        """Override parent — classify soil strength from phi."""
        if self.friction_angle < 20:
            return "Soft Clay"
        elif self.friction_angle < 28:
            return "Stiff Clay / Loose Sand"
        elif self.friction_angle < 36:
            return "Medium Dense Sand"
        elif self.friction_angle < 42:
            return "Dense Sand"
        else:
            return "Very Dense Sand / Gravel"


# Try both new child classes
cpt = CPTTest("BH-02", 4.5, qc=12.5, fs=45.0)
tri = TriaxialTest("BH-03", 8.0, sigma3=200, deviator_stress=350,
                   cohesion=15, friction_angle=32)

print("CPT Test:")
print(f"  Summary:  {cpt.summary()}")            # inherited
print(f"  Rf:       {cpt.friction_ratio()}%")     # CPT-specific
print(f"  Classify: {cpt.classify()}")            # overridden
print(f"  Analyze:  {cpt.analyze()}")
print()
print("Triaxial Test:")
print(f"  Summary:  {tri.summary()}")             # inherited
print(f"  sigma1:   {tri.sigma1()} kPa")          # triaxial-specific
print(f"  tau(100): {tri.shear_strength(100)} kPa") # Mohr-Coulomb at 100 kPa
print(f"  Classify: {tri.classify()}")             # overridden
print(f"  Analyze:  {tri.analyze()}")

---

## Part 4: Method Overriding in Action

**Method overriding** means a child class provides its own version of a method that already exists in the parent.

| Class | `analyze()` returns | `classify()` returns |
|-------|--------------------|-----------------------|
| `GeotechTest` | `"No analysis defined"` | `"Unclassified"` |
| `SPTTest` | N60, Vs, depth | Relative density (Very Loose…Very Dense) |
| `CPTTest` | qc, fs, Rf, depth | Robertson soil type |
| `TriaxialTest` | σ3, σ1, c, φ, depth | Strength class |

The method **name** is the same. The **behavior** depends on the object’s class. This is the foundation of polymorphism.

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD OVERRIDING — Same method name, different behavior
# ─────────────────────────────────────────────────────

# Three test objects — same method names, different results
spt1 = SPTTest("BH-01", 3.0, n_value=8)
spt2 = SPTTest("BH-01", 9.0, n_value=35)
cpt1 = CPTTest("BH-02", 5.0, qc=8.2, fs=120.0)
tri1 = TriaxialTest("BH-03", 7.0, sigma3=150, deviator_stress=280,
                    cohesion=10, friction_angle=28)

tests = [spt1, spt2, cpt1, tri1]

print(f"{'Test Type':16s} {'Location':20s} {'Classification':25s}")
print("─" * 65)
for t in tests:
    test_type = type(t).__name__
    print(f"{test_type:16s} {t.summary():20s} {t.classify():25s}")

print()
print("Notice: .classify() was called on ALL objects, but each class")
print("responded with its own logic. That is method overriding.")

---

## Part 5: Polymorphism — One Loop, Many Types

**Polymorphism** = "many forms." You call the **same method** on objects of **different types**, and each responds appropriately.

```python
for test in all_tests:     # list of mixed SPT, CPT, Triaxial
    test.analyze()          # each class runs its OWN version
    test.classify()         # no if/elif checking which type!
```

Without polymorphism, you would need:
```python
if isinstance(test, SPTTest):     classify_spt(test)
elif isinstance(test, CPTTest):   classify_cpt(test)
elif isinstance(test, Triaxial):  classify_triaxial(test)
```

With polymorphism, every new test type **just works** — no changes to the processing code.

In [ ]:
# ─────────────────────────────────────────────────────
# POLYMORPHISM — Process mixed test data uniformly
# ─────────────────────────────────────────────────────

# A realistic site investigation — mixed test types
site_tests = [
    SPTTest("BH-01", 1.5, n_value=5),
    SPTTest("BH-01", 3.0, n_value=12),
    SPTTest("BH-01", 6.0, n_value=28),
    SPTTest("BH-01", 9.0, n_value=45),
    CPTTest("BH-02", 2.0, qc=3.5,  fs=80.0),
    CPTTest("BH-02", 5.0, qc=12.0, fs=40.0),
    CPTTest("BH-02", 8.0, qc=18.5, fs=55.0),
    TriaxialTest("BH-03", 4.0, sigma3=100, deviator_stress=180,
                 cohesion=25, friction_angle=22),
    TriaxialTest("BH-03", 8.0, sigma3=250, deviator_stress=420,
                 cohesion=12, friction_angle=34),
]

# Polymorphic processing — ONE loop handles all types
print("Site Investigation Results")
print("=" * 75)
print(f"{'#':>2} {'Type':12s} {'BH':6s} {'Depth':>5s} {'Classification':25s}")
print("─" * 75)

for i, test in enumerate(site_tests, 1):
    tname = type(test).__name__.replace("Test", "")
    print(f"{i:2d} {tname:12s} {test.borehole_id:6s} {test.depth:4.1f}m "
          f"{test.classify():25s}")

print("─" * 75)
print(f"Total tests processed: {len(site_tests)}")
print()
print("Key insight: the loop does NOT check the type of each test.")
print("It just calls .classify() — and each class knows what to do.")

---

## Part 6: `isinstance()` and Type Checking

Sometimes you DO need to know the type — for example, to filter only SPT tests, or to count how many CPT readings you have.

| Function | What It Checks | Example |
|----------|---------------|---------|
| `isinstance(obj, Class)` | Is obj an instance of Class (or its child)? | `isinstance(spt, GeotechTest)` → `True` |
| `type(obj)` | The exact class of obj | `type(spt)` → `<class 'SPTTest'>` |
| `type(obj).__name__` | Class name as string | `type(spt).__name__` → `"SPTTest"` |

**Important:** `isinstance()` respects inheritance. An `SPTTest` IS a `GeotechTest`.

In [ ]:
# ─────────────────────────────────────────────────────
# isinstance() AND type() — Type checking with inheritance
# ─────────────────────────────────────────────────────

spt = SPTTest("BH-01", 6.0, n_value=18)
cpt = CPTTest("BH-02", 4.5, qc=12.5, fs=45.0)

# isinstance checks the inheritance chain
print("isinstance checks:")
print(f"  spt is SPTTest?     {isinstance(spt, SPTTest)}")       # True
print(f"  spt is GeotechTest? {isinstance(spt, GeotechTest)}")   # True!
print(f"  spt is CPTTest?     {isinstance(spt, CPTTest)}")       # False
print(f"  cpt is GeotechTest? {isinstance(cpt, GeotechTest)}")   # True
print()

# type() checks the EXACT class
print("type() checks:")
print(f"  type(spt) == SPTTest:     {type(spt) == SPTTest}")       # True
print(f"  type(spt) == GeotechTest: {type(spt) == GeotechTest}")   # False!
print()

# Practical: filter tests by type from a mixed list
all_tests = site_tests  # from Part 5

spt_only = [t for t in all_tests if isinstance(t, SPTTest)]
cpt_only = [t for t in all_tests if isinstance(t, CPTTest)]
tri_only = [t for t in all_tests if isinstance(t, TriaxialTest)]

print(f"Test counts from {len(all_tests)} total tests:")
print(f"  SPT tests:      {len(spt_only)}")
print(f"  CPT tests:      {len(cpt_only)}")
print(f"  Triaxial tests: {len(tri_only)}")
print()

# Compute average N60 from SPT tests only
avg_n60 = sum(t.corrected_n() for t in spt_only) / len(spt_only)
print(f"Average N60 (SPT only): {avg_n60:.1f}")

---

## Part 7: Multiple Inheritance — Mixins for Shared Capabilities

A class can inherit from **two or more parents**. This is useful for adding cross-cutting capabilities — like making any class printable or exportable.

```
Printable (mixin)    Analyzable (mixin)
    \                   /
     \                 /
      SiteReport  ← inherits from both
```

A **mixin** is a small class designed to be combined with others — it adds one specific capability without being used on its own.

In [ ]:
# ─────────────────────────────────────────────────────
# MULTIPLE INHERITANCE — Mixins for shared capabilities
# ─────────────────────────────────────────────────────

class Printable:
    """Mixin: adds formatted printing capability."""
    def print_report(self):
        print(f"\n{'=' * 50}")
        print(f"  REPORT: {type(self).__name__}")
        print(f"{'=' * 50}")
        for key, val in self.__dict__.items():
            if isinstance(val, float):
                print(f"  {key:20s}: {val:.2f}")
            else:
                print(f"  {key:20s}: {val}")
        print(f"{'=' * 50}")


class Exportable:
    """Mixin: adds data export capability."""
    def to_dict(self):
        return {"class": type(self).__name__, **self.__dict__}

    def to_csv_row(self):
        values = [str(v) for v in self.__dict__.values()]
        return ",".join(values)


# A class that inherits from GeotechTest AND both mixins
class EnhancedSPT(SPTTest, Printable, Exportable):
    """SPT test with print and export capabilities."""
    pass  # No new code needed — it inherits everything!


# Create an enhanced SPT test
espt = EnhancedSPT("BH-01", 6.0, n_value=18)

# From SPTTest (via GeotechTest)
print(f"Summary:  {espt.summary()}")
print(f"Classify: {espt.classify()}")
print(f"N60:      {espt.corrected_n()}")

# From Printable mixin
espt.print_report()

# From Exportable mixin
print(f"\nAs dict: {espt.to_dict()}")
print(f"CSV row: {espt.to_csv_row()}")

# Check the inheritance chain (Method Resolution Order)
print(f"\nMRO: {[c.__name__ for c in EnhancedSPT.__mro__]}")

---

## Part 8: Inheritance Hierarchy — Visualizing the Relationships

Let us verify the full class hierarchy and understand how Python resolves method calls.

```
              object
                │
           GeotechTest         Printable     Exportable
          /    |     \            |              |
    SPTTest  CPTTest  TriaxialTest   \          /
       |                              \        /
   EnhancedSPT  ───────────────────────
```

Python uses the **Method Resolution Order (MRO)** to decide which method to call when multiple parents define the same method.

In [ ]:
# ─────────────────────────────────────────────────────
# INHERITANCE HIERARCHY — issubclass and MRO
# ─────────────────────────────────────────────────────

# issubclass checks class relationships (not instances)
print("issubclass checks:")
print(f"  SPTTest < GeotechTest?      {issubclass(SPTTest, GeotechTest)}")
print(f"  CPTTest < GeotechTest?      {issubclass(CPTTest, GeotechTest)}")
print(f"  TriaxialTest < GeotechTest? {issubclass(TriaxialTest, GeotechTest)}")
print(f"  SPTTest < CPTTest?          {issubclass(SPTTest, CPTTest)}")
print(f"  EnhancedSPT < Printable?   {issubclass(EnhancedSPT, Printable)}")
print(f"  EnhancedSPT < Exportable?  {issubclass(EnhancedSPT, Exportable)}")
print()

# Show all parent classes
classes = [SPTTest, CPTTest, TriaxialTest, EnhancedSPT]
print("Parent classes (__bases__):")
for cls in classes:
    parents = [p.__name__ for p in cls.__bases__]
    print(f"  {cls.__name__:16s} inherits from: {parents}")
print()

# Full MRO for EnhancedSPT
print("Method Resolution Order (EnhancedSPT):")
for i, cls in enumerate(EnhancedSPT.__mro__):
    print(f"  {i}. {cls.__name__}")

---

## Part 9: Complete Application — Site Investigation Report System

Now let us put it all together. We will:
1. Create a collection of tests from three boreholes
2. Process them **polymorphically** through a single interface
3. Generate per-borehole and cross-site summaries
4. Use `isinstance()` to compute type-specific statistics

This is how real geotechnical software organizes data — different test types, one processing pipeline.

In [ ]:
# ═════════════════════════════════════════════════════
# COMPLETE APPLICATION: Site Investigation Report System
# ═════════════════════════════════════════════════════

import math

# ---- Site Investigation Database ----
investigation = [
    # BH-01: SPT tests at multiple depths
    SPTTest("BH-01",  1.5, n_value=4),
    SPTTest("BH-01",  3.0, n_value=8),
    SPTTest("BH-01",  4.5, n_value=14),
    SPTTest("BH-01",  6.0, n_value=22),
    SPTTest("BH-01",  9.0, n_value=35),
    SPTTest("BH-01", 12.0, n_value=50),

    # BH-02: CPT soundings
    CPTTest("BH-02",  1.0, qc=2.0,  fs=60.0),
    CPTTest("BH-02",  2.5, qc=5.5,  fs=85.0),
    CPTTest("BH-02",  4.0, qc=8.0,  fs=50.0),
    CPTTest("BH-02",  6.0, qc=14.0, fs=42.0),
    CPTTest("BH-02",  8.0, qc=20.0, fs=60.0),

    # BH-03: Triaxial tests on undisturbed samples
    TriaxialTest("BH-03", 3.0, sigma3=75,  deviator_stress=120,
                 cohesion=30, friction_angle=18),
    TriaxialTest("BH-03", 6.0, sigma3=150, deviator_stress=260,
                 cohesion=20, friction_angle=26),
    TriaxialTest("BH-03", 9.0, sigma3=250, deviator_stress=450,
                 cohesion=10, friction_angle=35),
]

print(f"Total tests in investigation: {len(investigation)}")
print(f"Boreholes: {sorted(set(t.borehole_id for t in investigation))}")

In [ ]:
# ─────────────────────────────────────────────────────
# UNIFIED SITE REPORT — Polymorphic processing
# ─────────────────────────────────────────────────────

print("=" * 80)
print("  SITE INVESTIGATION REPORT")
print("  Highway Bridge Foundation — Salt Lake City, Utah")
print("=" * 80)

# Group tests by borehole
boreholes = {}
for test in investigation:
    bh = test.borehole_id
    if bh not in boreholes:
        boreholes[bh] = []
    boreholes[bh].append(test)

# Process each borehole
for bh_id, tests in sorted(boreholes.items()):
    test_type = type(tests[0]).__name__.replace("Test", "")
    print(f"\n  {bh_id} — {test_type} Tests ({len(tests)} readings)")
    print(f"  {'─' * 72}")

    for test in tests:
        result = test.analyze()
        classification = test.classify()

        # Format depends on test type, but we get results polymorphically
        if isinstance(test, SPTTest):
            print(f"    {test.depth:5.1f}m | N60={result['N60']:3d} | "
                  f"Vs={result['Vs']:5.1f} m/s | {classification}")
        elif isinstance(test, CPTTest):
            print(f"    {test.depth:5.1f}m | qc={result['qc_MPa']:5.1f} MPa | "
                  f"Rf={result['Rf_%']:4.2f}% | {classification}")
        elif isinstance(test, TriaxialTest):
            print(f"    {test.depth:5.1f}m | c={result['c_kPa']:3d} kPa | "
                  f"\u03c6={result['phi_deg']:2d}\u00b0 | {classification}")

print(f"\n{'=' * 80}")

In [ ]:
# ─────────────────────────────────────────────────────
# CROSS-SITE STATISTICS — Type-specific analysis
# ─────────────────────────────────────────────────────

# Filter by type using isinstance
spt_tests = [t for t in investigation if isinstance(t, SPTTest)]
cpt_tests = [t for t in investigation if isinstance(t, CPTTest)]
tri_tests = [t for t in investigation if isinstance(t, TriaxialTest)]

print("CROSS-SITE STATISTICS")
print("=" * 60)

# SPT statistics
n60_values = [t.corrected_n() for t in spt_tests]
vs_values = [t.estimate_vs() for t in spt_tests]
print(f"\n  SPT Tests ({len(spt_tests)} total):")
print(f"    N60 range:    {min(n60_values)} \u2013 {max(n60_values)}")
print(f"    Avg N60:      {sum(n60_values)/len(n60_values):.1f}")
print(f"    Avg Vs:       {sum(vs_values)/len(vs_values):.1f} m/s")

# CPT statistics
qc_values = [t.qc for t in cpt_tests]
rf_values = [t.friction_ratio() for t in cpt_tests]
print(f"\n  CPT Tests ({len(cpt_tests)} total):")
print(f"    qc range:     {min(qc_values):.1f} \u2013 {max(qc_values):.1f} MPa")
print(f"    Avg Rf:       {sum(rf_values)/len(rf_values):.2f}%")

# Triaxial statistics
phi_values = [t.friction_angle for t in tri_tests]
c_values = [t.cohesion for t in tri_tests]
print(f"\n  Triaxial Tests ({len(tri_tests)} total):")
print(f"    \u03c6 range:      {min(phi_values)}\u00b0 \u2013 {max(phi_values)}\u00b0")
print(f"    Avg cohesion: {sum(c_values)/len(c_values):.1f} kPa")

# Overall classification summary
print(f"\n  Classification Summary:")
class_counts = {}
for test in investigation:
    cls = test.classify()
    class_counts[cls] = class_counts.get(cls, 0) + 1

for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    bar = "\u2588" * count
    print(f"    {cls:28s} {count:2d}  {bar}")

print(f"\n{'=' * 60}")

---

## Practice Exercises

### Exercise 1: Add a new child class
Create a `VaneShearTest` class that inherits from `GeotechTest`. It should store `su` (undrained shear strength in kPa) and override `classify()` to return consistency ("Very Soft" for su<12.5, "Soft" for su<25, "Firm" for su<50, "Stiff" for su<100, "Very Stiff" for su<200, else "Hard").

### Exercise 2: Polymorphic processing
Create a list with 2 SPTTest, 2 CPTTest, and 1 VaneShearTest. Loop through all five and call `.summary()`, `.classify()`, and `.analyze()` without checking the type.

### Exercise 3: Extend with a mixin
Create a `Loggable` mixin class that has a `log()` method printing `"[LOG] ClassName: summary()"`. Then create a `LoggedSPT(SPTTest, Loggable)` class and verify that `.log()`, `.classify()`, and `.summary()` all work.

In [ ]:
# EXERCISE 1
# class VaneShearTest(GeotechTest):
#     def __init__(self, borehole_id, depth, su, date="2026-02-27"):
#         super().__init__(borehole_id, depth, date)
#         self.su = su
#
#     def analyze(self):
#         ...
#
#     def classify(self):
#         ...

In [ ]:
# EXERCISE 2
# mixed_tests = [
#     SPTTest("BH-01", 3.0, n_value=10),
#     SPTTest("BH-01", 6.0, n_value=30),
#     CPTTest("BH-02", 4.0, qc=8.0, fs=50.0),
#     CPTTest("BH-02", 7.0, qc=15.0, fs=35.0),
#     VaneShearTest("BH-03", 5.0, su=45),
# ]
# for t in mixed_tests:
#     print(f"{type(t).__name__:16s} {t.summary():20s} {t.classify()}")

In [ ]:
# EXERCISE 3
# class Loggable:
#     def log(self):
#         print(f"[LOG] {type(self).__name__}: {self.summary()}")
#
# class LoggedSPT(SPTTest, Loggable):
#     pass
#
# ls = LoggedSPT("BH-01", 4.5, n_value=15)
# ls.log()
# print(ls.classify())

---

## Key Takeaways

1. **Inheritance** (`class Child(Parent)`) lets child classes reuse parent attributes and methods — write common logic once in `GeotechTest`, specialize in `SPTTest`, `CPTTest`, `TriaxialTest`.

2. **`super().__init__()`** calls the parent constructor so the child gets all base attributes (`borehole_id`, `depth`, `date`) without duplicating code.

3. **Method overriding** lets each child class replace parent methods (`analyze()`, `classify()`) with its own logic — same interface, different behavior.

4. **Polymorphism** means you can process a list of mixed test types in one loop by calling the same method names — no type-checking branches needed.

5. **`isinstance()`** respects inheritance (an `SPTTest` IS a `GeotechTest`), while `type()` checks the exact class. Use `isinstance()` for filtering, `type()` for exact matching.

6. **Multiple inheritance / Mixins** add cross-cutting capabilities (printing, exporting) without modifying the main class hierarchy.

---

## How This Connects to ML

| Today’s Concept | ML Equivalent |
|----------------|---------------|
| Base class `GeotechTest` | `sklearn.base.BaseEstimator` — all models inherit from it |
| `.analyze()` / `.classify()` override | `.fit()` / `.predict()` — every model implements its own version |
| Polymorphism (one loop, many types) | `for model in [SVM(), RF(), XGB()]: model.fit(X, y)` |
| `isinstance()` type checking | Checking if a model is a classifier vs regressor |
| Mixins (`Printable`, `Exportable`) | `ClassifierMixin`, `RegressorMixin` in scikit-learn |
| `super().__init__()` | Custom model `__init__` calling parent to register hyperparameters |
| Method Resolution Order (MRO) | How Python decides which `.fit()` to call in complex hierarchies |

When you write `model.fit(X, y)` in scikit-learn, you are using **exactly** the same pattern we built today: a base class defines the interface, each model overrides it, and your training loop processes them polymorphically.

---

*Day 011 of 100 — Inheritance builds families of classes; polymorphism lets them work together!*